<img src = "https://drive.google.com/uc?export=view&id=1Md0grlXXV4fB0bkYuMWzBiS64RvjaFGX" alt = "Encabezado MLDS" width = "100%">  </img>

# **Entendimiento del Negocio y Carga de Datos**
---

Este notebook es una plantilla que le puede servir como guía para el primer entregable del proyecto aplicado.

## **1. Marco de Proyecto**
---

Normalmente, se suele construir un marco de proyecto para mostrar los resultados del entendimiento del negocio, es decir, debemos dar respuesta a los siguientes elementos:

### **1.1. Trasfondo del Negocio**
---

- ¿Quién es el cliente o los beneficiarios del proyecto? ¿En qué dominio se encuentran (marketing, medicina, entre otros)?
- ¿Qué problemas del negocio o del dominio estamos tratando de solucionar?

- **Cliente/beneficiarios:** equipos de operación y seguridad (Centros de operaciones SOC/CSIRT y personal de ciberseguridad) que monitorean intrusiones y anomalías en redes IT (entornos corporativos) y OT (SCADA, sensores y redes industriales).
- **Dominio:** registro de eventos de seguridad (firewall, protocolos de red, tipos de log) que describen interacciones entre `source_ip` y `dest_ip` y el tipo de acción (`allowed/blocked`).
- **Problema del negocio:** el volumen de logs es alto (millones de registros). La revisión manual no escala y genera una gran cantidad de alertas sin priorización adecuada, aumentando el trabajo operativo y la probabilidad de pasar por alto incidentes.
- **Objetivo del negocio:** automatizar la detección y clasificación de eventos “inusuales” para **reducir falsos positivos**, **priorizar alertas** (qué revisar primero) y habilitar un esquema de análisis **casi en tiempo real** para responder con rapidez ante posibles amenazas.
- **Relación con ML/NLP:** además de señales estructuradas (IP, protocolo, acción, bytes), existen campos textuales como `user_agent` y `request_path` que pueden capturar patrones lingüísticos/técnicos relevantes para diferenciar tráfico normal vs. amenaza.

### **1.2. Alcance**
---

- ¿Qué  solución basada en _Deep Learning_ queremos implementar?
- ¿Qué  se hará?
- ¿De qué forma el cliente o beneficiario utilizará el producto del proyecto?

**Objetivo del proyecto:** construir un sistema basado en **Machine Learning + NLP** para detectar, clasificar y reportar anomalías en registros de logs de ciberseguridad, con foco en **entornos IT y OT**, con el fin de **reducir falsos positivos** y **priorizar alertas**.

La solución propone construir un sistema basado en **Machine Learning + NLP** para detectar, clasificar y reportar anomalías en registros de logs de ciberseguridad, con foco en **entornos IT y OT**.

- **Qué solución se implementará (alto nivel):**
  - Un **pipeline de ingesta y normalización** de logs a escala (millones de registros) usando **Spark**.
  - Un enfoque de **detección de anomalías** usando la columna `threat_label` como referencia de amenaza: se modela lo “normal” como `threat_label = benign` y lo “anómalo/amenaza” como `threat_label != benign`.
  - **Clasificación** del tipo de amenaza (multi-clase) cuando existan categorías en `threat_label`.
- **Qué se hará en esta entrega (Fase 1):**
  - Definir el objetivo del proyecto.
  - Identificar la(s) fuente(s) de datos y cargar el dataset en el entorno.
  - Verificar **calidad** (esquema, nulos, parseo de timestamps, rangos) y **cantidad** (número de registros).
  - Documentar el proceso de carga y la preparación inicial de datos para futuras fases.
- **Cómo lo usará el beneficiario (resultado esperado):**
  - Alertas/Reportes que prioricen eventos relevantes (menos falsos positivos) y soporten una revisión eficiente por analistas.
  - Estructura de salida para un posible modo “casi en tiempo real” (al disponer de un flujo de ingesta en batch/streaming en fases siguientes).

### **1.3. Plan**
---

Puede agregar una lista de actividades con tiempos estimados, un diagrama de Gantt o integrar alguna herramienta de gestión de proyectos para mostrar la planeación del proyecto.


**Planeación propuesta:**

| Semana (Fase) | Enfoque | Actividades clave (ML/NLP) | Resultado/Entregable |
|---|---|---|---|
| Semana 1 (Fase 1) | Entendimiento del negocio + carga | Definir labels/objetivos: `is_anomaly` desde `threat_label` (benign vs amenaza). Preparar el “contrato de datos” para ML: tipos finales, normalización mínima y columnas candidatas de features (numéricas/categóricas/textuales). Justificar el muestreo: procesar `50.000` filas para ejecución, pero diseñar pipeline que escale a ~`6MM` (transformaciones lazy y esquema explícito). | Dataset listo para modelado + evidencias de calidad (nulos/vacíos, timestamp parseado, balance de clases). |
| Semana 2 (Fase 2) | Preparación de features + NLP | Ingeniería de features: (1) numéricas: `bytes_transferred` (log-transform si aplica, clipping), (2) categóricas: `protocol`, `action`, `log_type` (one-hot o hashing), (3) IPs: codificación por hashing/embedding ligero (evitar cardinalidad extrema). NLP: preprocesar `user_agent` y `request_path` (lower/trim + limpiar caracteres no informativos) y convertir a vectores con `Tokenizer` + `HashingTF/IDF` (word- o char-n-grams vía splitting por delimitadores como `/`, espacios y símbolos). Definir `VectorAssembler` y estructura final `features` + labels para MLlib. | Pipeline de preparación reproducible: columnas `features` + `label`/`is_anomaly` + split 70/30. |
| Semana 3 (Fase 2) | Baseline supervisado + detección | Baseline de clasificación (recomendado para reducir falsos positivos al calibrar): entrenar modelos supervisados con `is_anomaly` (y opcionalmente multi-clase con `threat_label`) usando MLlib (p. ej. `LogisticRegression`, `RandomForestClassifier` o `GBTClassifier`). Comparar con una estrategia “detección” entrenando solo con `threat_label=benign` (baseline de probabilidad/score por similitud o umbrales sobre el modelo) para obtener un `anomaly_score`. Evaluación inicial con métricas clave para desbalance: precision/recall/F1 para la clase anómala, ROC-AUC y PR-AUC, y matriz de confusión. | Baselines entrenados + reporte de métricas + análisis de errores (qué tipos de amenazas causan más FP/FN). |
| Semana 4 (Fase 3) | Calibración para reducir falsos positivos | Ajuste orientado al objetivo: calibrar umbrales para transformar `prob(anomalía)` en alerta (maximizar F-beta o limitar tasa de falsos positivos). Si se tiene `anomaly_score`, definir estrategia de “severidad/prioridad” (ranking por score o por combinación score + severidad del tipo de amenaza). Realizar validación por partición (evitar leakage por split determinista) y repetir evaluación. En NLP, iterar tokenización (word vs char-n-grams, regex de limpieza) y medir impacto sobre FP. | Umbrales/estrategia de priorización + justificación cuantitativa (mejor trade-off FP vs recall). |
| Semana 5 (Cierre) | Validación final + preparación de despliegue futuro | Consolidar resultados del mejor pipeline: set final de métricas y curvas (PR-AUC), reportar interpretabilidad operativa (top tokens/patrones en `request_path`/`user_agent` vía inspección de features o análisis de tokens más frecuentes por clase). Documentar cómo operar “casi en tiempo real”: transformar el pipeline de features en un flujo aplicable a batch/streaming, y definir ventana de agregación y salida de alertas. | Informe final: visión ML/NLP, lecciones aprendidas y plan para evolución a producción (incluye adaptación a IT/OT). |

## **2. Datos**
---

En esta sección debe adjuntar una descripción de los datos y agregar el código que permite su tratamiento.

### **2.1. Origen**
---

- ¿De dónde vienen los datos?
- ¿Se usa alguna herramienta o proceso para la descarga de la información?
- ¿Qué tipo de datos estamos manejando?

### **2.1. Origen**

- **Fuente principal:** el dataset de Kaggle *“Cybersecurity Threat Detection Logs”* descargado vía `kagglehub` y/o almacenado localmente en la raíz del proyecto.
- **Tipo de datos:** eventos de red y seguridad con campos estructurados y textuales.

**Columnas relevantes (según el CSV):** `timestamp`, `source_ip`, `dest_ip`, `protocol`, `action`, `threat_label`, `log_type`, `bytes_transferred`, `user_agent`, `request_path`.

### **2.1.1 Estrategia de fuentes y formatos (para IT y OT)**

En fases posteriores se integrarán fuentes adicionales (por ejemplo, logs de SIEM/NDR/EDR) con formatos variados (CSV/JSON/Parquet). Esta fase documenta la estrategia base para que el pipeline sea extensible.

### **2.1.2 Criterios de calidad y cantidad (qué validaremos)**

- **Cantidad:** conteo total de filas para confirmar la carga correcta a escala (millones de registros).
- **Calidad del esquema:** presencia de columnas esperadas y tipado interpretable (por ejemplo `bytes_transferred` numérico; `timestamp` parseable).
- **Integridad:** conteo de nulos/vacíos por columnas clave (`timestamp`, `source_ip`, `dest_ip`, `protocol`, `action`, `threat_label`, `log_type`, `bytes_transferred`, `user_agent`, `request_path`).
- **Consistencia contextual mínima:**
  - `timestamp` debe convertirse a timestamp real.
  - `bytes_transferred` debe ser no-negativo (los inválidos se tratarán en limpieza posterior).
- **Calidad semántica disponible:** `threat_label` actúa como referencia supervisable. Se usará para definir anomalía operacionalmente como `threat_label != 'benign'` y medir el balance de clases.

Con estas validaciones se garantiza que el dataset queda listo para reducir falsos positivos y habilitar priorización de alertas en fases posteriores.

Para agilizar la ejecución de la entrega (Fase 1), el notebook procesa una muestra de `50.000` registros; sin embargo, el dataset completo tiene aproximadamente `6MM` registros (se reporta para mantener el contexto de escala).

Adicionalmente, el notebook guarda/carga la muestra en formato Parquet para evitar re-procesar el CSV completo en corridas posteriores.

### **2.2. Carga o Adquisición de Datos**
---

Agregue el código necesario en _Python_ para obtener o cargar los datos.

In [1]:
# Carga y verificación de datos (escala: millones de registros) con Spark

from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, LongType
from pyspark.sql.functions import (
    col,
    to_timestamp,
    when,
    trim,
    lower,
    lit,
    sum as spark_sum,
    count as spark_count,
    xxhash64,
)

# -----------------------------
# 1) Configuración
# -----------------------------
from pathlib import Path

DATA_FILENAME = "cybersecurity_threat_detection_logs.csv"
DATA_PATH = Path(DATA_FILENAME)  # se asume que el CSV está en la misma carpeta que el notebook

# Si el archivo no existe localmente, intenta descargarlo con kagglehub (opcional).
if not DATA_PATH.exists():
    try:
        import kagglehub

        # Dataset de Kaggle según tu indicación.
        downloaded_path = kagglehub.dataset_download("aryan208/cybersecurity-threat-detection-logs")
        downloaded_root = Path(downloaded_path)

        matches = list(downloaded_root.rglob(DATA_FILENAME))
        if matches:
            DATA_PATH = matches[0]
        else:
            raise FileNotFoundError(
                f"No se encontró {DATA_FILENAME} dentro de la carpeta descargada por kagglehub: {downloaded_root}"
            )
    except ImportError as e:
        raise FileNotFoundError(
            f"No existe {DATA_FILENAME} en la carpeta del notebook y no está disponible kagglehub. "
            "Coloca el CSV en la carpeta del notebook o instala kagglehub."
        ) from e

spark = SparkSession.builder.appName("M5U2_1_business_data_load").getOrCreate()

# Esquema explícito para evitar inferencia costosa y garantizar consistencia.
schema = StructType(
    [
        StructField("timestamp", StringType(), True),
        StructField("source_ip", StringType(), True),
        StructField("dest_ip", StringType(), True),
        StructField("protocol", StringType(), True),
        StructField("action", StringType(), True),
        StructField("threat_label", StringType(), True),
        StructField("log_type", StringType(), True),
        StructField("bytes_transferred", LongType(), True),
        StructField("user_agent", StringType(), True),
        StructField("request_path", StringType(), True),
    ]
)

# -----------------------------
# 2) Carga
# -----------------------------
print("Cargando CSV con Spark desde:", DATA_PATH)

df_raw = (
    spark.read.schema(schema)
    .option("header", "true")
    .option("mode", "PERMISSIVE")
    .csv(str(DATA_PATH))
)

# -----------------------------
# 3) Normalización inicial
# -----------------------------
# - Parsear timestamp
# - Normalizar strings (trim/lower) para reducir variación
# - Definir etiqueta de anomalía (ground truth disponible): threat_label != 'benign'

df = df_raw

df = df.withColumn("timestamp_ts", to_timestamp(col("timestamp")))

def _norm_str(c):
    return lower(trim(col(c)))

for c in ["protocol", "action", "threat_label", "log_type", "source_ip", "dest_ip", "user_agent", "request_path"]:
    df = df.withColumn(c, _norm_str(c))

# Definición operacional de anomalía (basada en el dataset):
# - Normal: threat_label == benign
# - Anómalo/amenaza: threat_label != benign

df = df.withColumn(
    "is_anomaly",
    when(col("threat_label") != lit("benign"), lit(1)).otherwise(lit(0)),
)

# -----------------------------
# Submuestreo para ejecución rápida
# -----------------------------
# El dataset completo contiene aproximadamente 6MM registros, pero para esta entrega
# (Fase 1) se procesan solo 50.000 filas a fin de acelerar la ejecución del notebook.
DATASET_TOTAL_RECORDS_EST = 6_000_000
MAX_EXECUTION_ROWS = 50_000

print(f"Dataset completo (aprox.): {DATASET_TOTAL_RECORDS_EST} registros")
print(f"Procesando solo una muestra para la entrega: {MAX_EXECUTION_ROWS} registros")

# Nota: `limit` aplica un recorte en Spark; el objetivo aquí es hacer la entrega
# ejecutable rápido manteniendo las validaciones y normalización representativas.
#
# Para acelerar ejecuciones futuras, se puede guardar/cargar esta muestra en Parquet.
SAMPLE_TOTAL_RECORDS_EST = 50_000  # usado para documentación
SAMPLE_SAVE_DIR = "data_samples/muestra_50k_parquet"
USE_SAVED_SAMPLE = True
SAVE_SAMPLE_IF_MISSING = True

SAMPLE_SAVE_DIR_PATH = Path(SAMPLE_SAVE_DIR)

def _has_parquet_files(dir_path: Path) -> bool:
    # Verifica existencia de archivos Parquet dentro del directorio.
    if not dir_path.exists():
        return False
    return any(dir_path.rglob("*.parquet"))

if USE_SAVED_SAMPLE and _has_parquet_files(SAMPLE_SAVE_DIR_PATH):
    print(f"Cargando muestra guardada desde: {SAMPLE_SAVE_DIR_PATH}")
    df = spark.read.parquet(str(SAMPLE_SAVE_DIR_PATH))
else:
    print(f"Generando muestra para esta entrega: {MAX_EXECUTION_ROWS} filas")
    df = df.limit(MAX_EXECUTION_ROWS)
    if SAVE_SAMPLE_IF_MISSING:
        print(f"Guardando muestra en Parquet para futuras corridas: {SAMPLE_SAVE_DIR_PATH}")
        df.write.mode("overwrite").parquet(str(SAMPLE_SAVE_DIR_PATH))

# -----------------------------
# 4) Verificación de calidad y cantidad
# -----------------------------

# 4.1 Cantidad total
n_total = df.count()
print("Registros procesados (muestra para esta entrega):", n_total)

# 4.2 Parse correcto de timestamp
n_invalid_ts = (
    df.select(spark_sum(when(col("timestamp_ts").isNull(), 1).otherwise(0)).alias("invalid_ts"))
    .collect()[0]["invalid_ts"]
)
print("Registros con timestamp inválido:", n_invalid_ts)

# 4.3 Nulos/vacíos por columnas clave
# Para columnas string, contamos nulos y vacíos (trim == '').
string_cols = ["timestamp", "source_ip", "dest_ip", "protocol", "action", "threat_label", "log_type", "user_agent", "request_path"]

quality_aggs = []
for c in string_cols:
    quality_aggs.append(
        spark_sum(when(col(c).isNull() | (trim(col(c)) == lit("")), 1).otherwise(0)).alias(f"{c}__null_or_empty")
    )

# Para columnas numéricas, solo nulos
quality_aggs.append(spark_sum(when(col("bytes_transferred").isNull(), 1).otherwise(0)).alias("bytes_transferred__null"))

quality_df = df.select(*quality_aggs)
print("Chequeo de nulos/vacíos (columnas clave):")
quality_df.show(truncate=False)

# 4.4 Rango mínimo esperado (bytes_transferred >= 0)
invalid_bytes = (
    df.select(
        spark_sum(when(col("bytes_transferred").isNull() | (col("bytes_transferred") < lit(0)), 1).otherwise(0)).alias("invalid_bytes")
    )
    .collect()[0]["invalid_bytes"]
)
print("Registros con bytes_transferred inválido (nulo o negativo):", invalid_bytes)

# 4.5 Distribución de etiquetas (balance de clases)
print("Distribución de threat_label:")
df.groupBy("threat_label").agg(spark_count(lit(1)).alias("count")).orderBy(col("count").desc()).show(50, truncate=False)

# 4.6 Conteo de anomalías según definición is_anomaly
anomaly_counts = (
    df.groupBy("is_anomaly").agg(spark_count(lit(1)).alias("count")).orderBy("is_anomaly").collect()
)
print("Conteo de anomalías (definición: threat_label != 'benign'):")
for row in anomaly_counts:
    print(f"is_anomaly={row['is_anomaly']}: {row['count']}")

# -----------------------------
# 5) Partición 70/30 (base para fases posteriores)
# -----------------------------
# Para grandes volúmenes se prefiere una partición determinista (hash) sobre una aleatoria no reproducible.
# Esta partición se deja lista para que el entrenamiento/validación sea consistente.

df_split = df.withColumn(
    "split_id",
    (xxhash64(
        col("source_ip"),
        col("dest_ip"),
        col("protocol"),
        col("request_path"),
        col("timestamp_ts"),
    ) % lit(10)).cast("int"),
)

train_df = df_split.filter(col("split_id") < lit(7)).drop("split_id")
val_df = df_split.filter(col("split_id") >= lit(7)).drop("split_id")

print("Conteo train (70% aprox):", train_df.count())
print("Conteo valid (30% aprox):", val_df.count())

# -----------------------------
# 6) Muestra para inspección rápida
# -----------------------------
print("Muestra de registros normalizados:")
df.select(
    "timestamp_ts",
    "source_ip",
    "dest_ip",
    "protocol",
    "action",
    "threat_label",
    "log_type",
    "bytes_transferred",
    "user_agent",
    "request_path",
    "is_anomaly",
).show(5, truncate=False)


100%|██████████| 95.7M/95.7M [00:01<00:00, 85.0MB/s]

Extracting files...


Cargando CSV con Spark desde: /root/.cache/kagglehub/datasets/aryan208/cybersecurity-threat-detection-logs/versions/1/cybersecurity_threat_detection_logs.csv
Dataset completo (aprox.): 6000000 registros
Procesando solo una muestra para la entrega: 50000 registros
Generando muestra para esta entrega: 50000 filas
Guardando muestra en Parquet para futuras corridas: data_samples/muestra_50k_parquet
Registros procesados (muestra para esta entrega): 50000
Registros con timestamp inválido: 0
Chequeo de nulos/vacíos (columnas clave):
+------------------------+------------------------+----------------------+-----------------------+---------------------+---------------------------+-----------------------+-------------------------+---------------------------+-----------------------+
|timestamp__null_or_empty|source_ip__null_or_empty|dest_ip__null_or_empty|protocol__null_or_empty|action__null_or_empty|threat_label__null_or_empty|log_type__null_or_empty|user_agent__null_or_empty|request_path__null_

## **Créditos**

* **Profesor:** [Fabio Augusto Gonzalez](https://dis.unal.edu.co/~fgonza/)
* **Asistentes docentes :**
  * [Santiago Toledo Cortés](https://sites.google.com/unal.edu.co/santiagotoledo-cortes/)
* **Diseño de imágenes:**
    - [Mario Andres Rodriguez Triana](https://www.linkedin.com/in/mario-andres-rodriguez-triana-394806145/).
* **Coordinador de virtualización:**
    - [Edder Hernández Forero](https://www.linkedin.com/in/edder-hernandez-forero-28aa8b207/).

**Universidad Nacional de Colombia** - *Facultad de Ingeniería*